In [ ]:
import os
import pandas as pd

### Etapes ? suivre pour construire un mod?le de pr?diction:
- R?colter les donn?es
- Nettoyer et pr?parer les donn?es: elle permet d'aboutir ? un jeu de donn?es sur lequel entra?n? un algorithme
- Vectorisation: Repr?senter les mots en vecteurs
- Splitting du dataset en TestSet et TrainingSet
- Entra?n? et ?valuer
- Selectionner le meilleur mod?le

In [ ]:
os.chdir("F:/[Linkedin Learning] Machine learning  Traitement du langage naturel avec Python/fichiers_d_exercice_machine_learning_nlp")

In [ ]:
dataset = open("./Data/SMSSpamCollection.txt").read()
dataset = dataset.replace("\t",'\n')
dataset = dataset.split('\n')


In [ ]:
labels = dataset[0::2]
content = dataset[1::2]

In [ ]:
df = pd.DataFrame({
    'label':labels[:-1],
    'Content':content
})
df

In [ ]:
df.to_csv("./Data/Dataset.csv",sep=",",index=None)

In [ ]:
dataset = pd.read_csv("./Data/Dataset.csv")
dataset.head()

In [ ]:
nb_spam = len(dataset[dataset.label=='spam'])
nb_ham = len(dataset[dataset.label=='ham'])

print(f'Dans le dataset il y a \n{nb_spam} email spam \n{nb_ham} email ham')


## Regex

In [ ]:
import re

In [ ]:
test = "Je viens de commencer le NLP ce jour le 15/12/2023"

In [ ]:
re.findall('[a-z]+er',test)

## Data preprocessing

- Enlever les ponctuations
- Enlever les stops words
- Reduction de la taille d'un texte avec le stemming ou la lemmatization

#### Suppression des ponctuations

In [ ]:
import string
string.punctuation

In [ ]:
def Remove_Punctuation(text):
    result = "".join([ch for ch in text if ch not in string.punctuation])
    
    return result

In [ ]:
dataset['Content_cleaned'] = dataset['Content'].apply(lambda x: Remove_Punctuation(x))

In [ ]:
dataset.head()

#### Suppression des stop words

In [ ]:
def tokenizer(text):
    return re.split("\W+",text)


dataset['Content_tokenized'] = dataset['Content_cleaned'].apply(lambda x: tokenizer(x.lower()))

In [ ]:
dataset.head()

In [ ]:
import nltk

In [ ]:
en_stopwords = nltk.corpus.stopwords.words('english')

In [ ]:
def Remove_stopwords(content_list):
    return [word for word in content_list if word not in en_stopwords]

In [ ]:
dataset['Content_without_stopwords'] = dataset.Content_tokenized.apply(lambda x:Remove_stopwords(x))

In [ ]:
dataset.head()

### Stemming

- avantage: vitesse

- inconvenient: mauvaise compression, contexte non pris en compte ? cause des heuristiques
- Les mots g?n?r?s sont hors dictionnaire

In [ ]:
ps = nltk.PorterStemmer()

In [ ]:
def Stemming(words):
    return [ps.stem(word) for word in words]

dataset['Content_Stemmed'] = dataset.Content_without_stopwords.apply(lambda x:Stemming(x))

In [ ]:
dataset.head()

### Lemmatization

In [ ]:
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')
wn = WordNetLemmatizer()

In [ ]:
def Lemmatization(words):
    result = [wn.lemmatize(word) for word in words]
    return " ".join(result)

In [ ]:
dataset['Content_Lemmatized'] = dataset.Content_without_stopwords.apply(lambda x:Lemmatization(x))

In [ ]:
dataset.head(10)

### commentaires sur la lemmatisation
- Plus pr?cise dans la contraction
- Plus lente dans le traitement
- Les mots g?n?r?s sont pr?sent dans le dictionnaire utilis?

## Vectorization

Utilisation de la technique de comptage

In [ ]:
def clean_email(email):
    result = "".join([word for word in email if word not in string.punctuation])
    tokens = re.split("\W+",result)
    text = [wn.lemmatize(word) for word in tokens if word not in en_stopwords]
    return text

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer


vectorization_full = CountVectorizer(analyzer=clean_email)
vect_final = vectorization_full.fit_transform(dataset['Content'])
print(vect_final.shape)

## Vectorisation avec la technique N-Grams pour r?cup?rer le contexte

In [ ]:
ngram_vectorisation = CountVectorizer(ngram_range=(2,2))
## le param?tre ngram signifie qu'on compte les occurences deux ? deux

vect_final = ngram_vectorisation.fit_transform(dataset['Content_Lemmatized'])
print(vect_final.shape)


## Vectorisation avec TF-IDF (term Frequency-Inverse Document Frequency


$W_{i,j}=tf_{i,j}*\log(\dfrac{N}{df_{i}})$

$tf_{i,j}$: taux d'apparition d'un terme i dans le document j

$N$ nombre de documents total
$df_{i}$ normbre de document contenant le terme i

cette expression montre traduit $l'importance d'un document dans un document$

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tdif_vectoriszation = TfidfVectorizer()
tfidf_final = tdif_vectoriszation.fit_transform(dataset['Content_Lemmatized'])
print(tfidf_final.shape)

## Features engineering

### objectif:
- Enrichir les donn?es avec des informations utiles

### M?thodes
- Ajouter de nouvelles variables
- Transformer les variables existantes

#### Exemple de nouvelles variables pour le mod?le de SPAM
- Taux de caract?res de ponctuation
- Taille des messages
- Taux de lettre majuscules
- Pr?sence de certains mots tels que "lol"


#### Exemple de transformations de variables
- POW pour aboutir ? une amplication des des valeurs des variables
- Normalisation des variables pour mettre ? l'?chelle

In [ ]:
dataset['Content_len'] = dataset.Content.apply(lambda x: len(x)-x.count(" "))
dataset.Content_len

In [ ]:
def count_punctuation(text):
    binary_array = [1 for ch in text if ch in string.punctuation] ## comprehensive list
    nb_punctuation = sum(binary_array)
    total = len(text)-text.count(" ")
    return round(nb_punctuation/(total),4)*100

In [ ]:
dataset['punctuation_rate'] = dataset.Content.apply(lambda x:count_punctuation(x))
dataset.head()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
%matplotlib inline

In [ ]:
bins = np.linspace(0,200,40)
plt.hist(dataset[dataset['label']=='spam']['Content_len'],bins,alpha=0.5,label="spam")
plt.hist(dataset[dataset['label']=='ham']['Content_len'],bins,alpha=0.5,label="ham")
plt.legend(loc="upper right")
plt.show()

On remarque que la plupart des emails qui contient plus de 100 caract?res sont des spam

In [ ]:
bins = np.linspace(0,50,40)
plt.hist(dataset[dataset['label']=='spam']['punctuation_rate'],bins,alpha=0.5,label="spam")
plt.hist(dataset[dataset['label']=='ham']['punctuation_rate'],bins,alpha=0.5,label="ham")
plt.legend(loc="upper right")
plt.show()

#### Le taux de ponctuation n'influence pas sur la nature de l'email

## Comprendre l'exp?rience du machine learning

In [ ]:
dataset.shape

## K-Fold

Elle repose sur l'it?ration du sc?nario de splitting plusieurs fois en faisant varier les donn?es d'entrainement et de test. La performance du mod?le sera la moyenne des performances obtenues

### Matrice de confusion

Il faut d?terminer les cas positifs. Dans notre exemple, un cas positif est un email $spam$

### M?sures de performance

- Le recall = $\dfrac{VP}{VP+FN}$

le rapport entre les cas qui sont r?ellement positifs ( qui sont des spam ) et la totalit? des cas r?ellement positifs (vrai spam faux ham) 
elle correspond au taux de cas positifs rep?r? par le mod?le

- La pr?cision = $\dfrac{VP}{VP+FP}$

Le rapport entre les vrais spam et la somme des vrais spam et des faux spam
Elle renseigne sur le taux de vrais positifs d?tect?s

## Mod?lisation (SVM)



In [ ]:
""" Utilisation des donn?es vectoris?es avec le CountVectorizer et on cr?e un dataset
contenant les emails vectoris? et les variables ajout?es ( longueur et taux de ponctutation)
"""
all_data = pd.concat([pd.DataFrame(vect_final.toarray()),
                      dataset['Content_len'],
                      dataset['punctuation_rate']],axis=1)

In [ ]:
all_data

In [ ]:
from sklearn.metrics import precision_recall_fscore_support as score
from sklearn.model_selection import train_test_split

In [ ]:
## Split les donn?es en donn?es d'entrainement et de test
X_train,X_test,y_train,y_test = train_test_split(all_data,dataset['label'],test_size=.2)

In [ ]:
## Import du mod?le svm
from sklearn import svm

alg_svm = svm.SVC(kernel="linear")

In [ ]:
## entrainement du mod?le
alg_svm.fit(X_train,y_train)

In [ ]:
%%time
preds = alg_svm.predict(X_test)

In [ ]:
len(preds)

In [ ]:
precision,recall,fscore,_ = score(y_test,preds,pos_label="spam",average='binary')

In [ ]:
print(f"Precision : {precision*100:.2f} %\nRecall: {recall*100:.2f}% \nAccuracy : {fscore*100:.2f}%")

## Cross-validation

cherchons les performances globale du mod?le

In [ ]:
%%time
from sklearn.model_selection import KFold,cross_val_score

svm_with_cross_val = svm.SVC(kernel='linear')

""" S?paration des donn?es en 4 groupes (3 pour l'entrainement et 1 pour le test). On se concentre donc sur la mesure globale.
le param?tre n_jobs permet de v?rifier s'il est possible de faire du parall?lisme alors on fait les quatre ex?cution en m?me temps
"""


k_fold = KFold(n_splits=4)
cross_val_score(svm_with_cross_val,all_data,dataset['label'],cv=k_fold,scoring='accuracy',n_jobs=-1)

## Cross validation with TF-IDF

In [ ]:
all_data_tfidf = pd.concat([pd.DataFrame(tfidf_final.toarray()),
                      dataset['Content_len'],
                      dataset['punctuation_rate']],axis=1)

In [ ]:
## Split les donn?es en donn?es d'entrainement et de test
X_train,X_test,y_train,y_test = train_test_split(all_data_tfidf,dataset['label'],test_size=.2)

In [ ]:
alg_svm.fit(X_train,y_train)

In [ ]:
%%time
preds = alg_svm.predict(X_test)

In [ ]:
precision,recall,fscore,_ = score(y_test,preds,pos_label="spam",average='binary')
print(f"Precision : {precision*100:.2f} %\nRecall: {recall*100:.2f}% \nAccuracy : {fscore*100:.2f}%")

In [ ]:
%%time
from sklearn.model_selection import KFold,cross_val_score

svm_with_cross_val = svm.SVC(kernel='linear')

""" S?paration des donn?es en 4 groupes (3 pour l'entrainement et 1 pour le test). On se concentre donc sur la mesure globale.
le param?tre n_jobs permet de v?rifier s'il est possible de faire du parall?lisme alors on fait les quatre ex?cution en m?me temps
"""


k_fold = KFold(n_splits=4)
cross_val_score(svm_with_cross_val,all_data_tfidf,dataset['label'],cv=k_fold,scoring='accuracy',n_jobs=-1)

## Mod?lisation (Random Forest)

In [ ]:
from sklearn.ensemble import RandomForestClassifier as rfc

In [ ]:
rf_clf = rfc(n_estimators=50,max_depth=20,n_jobs=-1)
model = rf_clf.fit(X_train,y_train)

In [ ]:
preds_rf = model.predict(X_test)
precision,recall,fscore,_ = score(y_test,preds_rf,pos_label="spam",average='binary')
print(f"Precision : {precision*100:.2f} %\nRecall: {recall*100:.2f}% \nAccuracy : {fscore*100:.2f}%")

on remarque le mod?le de random forest est moins performant que le SVM. On remarque de seulement 62% des spam ont ?t? d?tect?s et tous ceux qui ont ?t? d?tect?s sont r?ellement des spam

### Cross validation with RandomForest

In [ ]:
%%time
from sklearn.model_selection import KFold,cross_val_score

rf_with_cross_val = rfc(n_estimators=50,max_depth=20,n_jobs=-1)

""" S?paration des donn?es en 4 groupes (3 pour l'entrainement et 1 pour le test). On se concentre donc sur la mesure globale.
le param?tre n_jobs permet de v?rifier s'il est possible de faire du parall?lisme alors on fait les quatre ex?cution en m?me temps
"""


k_fold = KFold(n_splits=4)
cross_val_score(rf_with_cross_val,all_data_tfidf,dataset['label'],cv=k_fold,scoring='accuracy',n_jobs=-1)